# Notebook 03 — Part 3: Autoencoder, Transfer Learning, and Backbone Model

This notebook covers the Part 3 work. The non-test data is already split into Block 1 and Block 2. I use Block 1 for representation learning with an autoencoder, then use Block 2 for age-group classification.

## 1. Setup and automatic data check

In [1]:


from pathlib import Path
import os, glob, shutil, subprocess

PROJECT_DIR = Path('/content/facial_age_project')
DATA_DIR = Path('/content/facial_age_dataset')
RAW_ZIP = Path('/content/facial-age.zip')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = 'frabbisw/facial-age'

if not Path('/root/.kaggle/kaggle.json').exists():
    print('Please upload kaggle.json when the upload button appears.')
    print('Kaggle > Account Settings > API > Create New Token')
    from google.colab import files
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('kaggle.json was not uploaded. Please upload the Kaggle API token file.')
    Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
    shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

if not DATA_DIR.exists() or not any(DATA_DIR.rglob('*')):
    print('Downloading the Facial Age dataset from Kaggle...')
    subprocess.run(['pip', '-q', 'install', 'kaggle'], check=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-p', '/content'], check=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', str(RAW_ZIP), '-d', str(DATA_DIR)], check=True)
    print('Dataset download and extraction complete.')
else:
    print('Dataset folder already exists in this Colab runtime.')

Please upload kaggle.json when the upload button appears.
Kaggle > Account Settings > API > Create New Token


Saving kaggle.json to kaggle.json
Dataset download and extraction complete.


In [2]:
# ============================================================
# MANIFEST CREATION FUNCTION
# ============================================================
# A manifest is a simple CSV table that stores each image path and its labels.
# Using CSV files makes the later training notebooks cleaner and avoids scanning
# all folders repeatedly.

import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
import os, glob

SEED = 42
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def age_to_group(age):
    """Map exact age into a small number of understandable age groups."""
    age = int(age)
    if age <= 12:
        return 'child'
    if age <= 19:
        return 'teen'
    if age <= 30:
        return 'youth'
    if age <= 45:
        return 'adult'
    if age <= 60:
        return 'mature'
    return 'senior'

def find_age_folders(root):
    """Find folders whose names are ages. This matches the Kaggle dataset layout."""
    root = Path(root)
    return sorted([p for p in root.rglob('*') if p.is_dir() and p.name.isdigit()], key=lambda x: int(x.name))

def build_manifests(project_dir=PROJECT_DIR, data_dir=DATA_DIR):
    project_dir = Path(project_dir)
    data_dir = Path(data_dir)
    project_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for age_folder in find_age_folders(data_dir):
        age = int(age_folder.name)
        for image_path in age_folder.iterdir():
            if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({
                    'image_path': str(image_path),
                    'age': age,
                    'age_group': age_to_group(age)
                })

    if not rows:
        raise RuntimeError('No images were found. Check that the Kaggle dataset was extracted correctly.')

    df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
    df.to_csv(project_dir / 'facial_age_manifest.csv', index=False)

    # The test set is separated first so it stays completely held out.
    train_val_df, test_df = train_test_split(
        df,
        test_size=0.10,
        random_state=SEED,
        stratify=df['age_group']
    )

    # The validation split is taken only from the remaining non-test data.
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=0.15,
        random_state=SEED,
        stratify=train_val_df['age_group']
    )

    # Part 3 needs two independent non-test blocks.
    block1_df, block2_df = train_test_split(
        train_val_df,
        test_size=0.50,
        random_state=SEED,
        stratify=train_val_df['age_group']
    )

    files = {
        'train_manifest.csv': train_df,
        'val_manifest.csv': val_df,
        'test_manifest.csv': test_df,
        'block1_autoencoder_manifest.csv': block1_df,
        'block2_classification_manifest.csv': block2_df,
    }

    for filename, frame in files.items():
        frame.to_csv(project_dir / filename, index=False)

    print('Manifest files created successfully:')
    for filename in ['facial_age_manifest.csv'] + list(files.keys()):
        print(' -', project_dir / filename)

    return df, train_df, val_df, test_df, block1_df, block2_df

def ensure_manifests():
    required = [
        PROJECT_DIR / 'train_manifest.csv',
        PROJECT_DIR / 'val_manifest.csv',
        PROJECT_DIR / 'test_manifest.csv',
        PROJECT_DIR / 'block1_autoencoder_manifest.csv',
        PROJECT_DIR / 'block2_classification_manifest.csv'
    ]
    if all(p.exists() for p in required):
        print('Manifest files already exist. Loading them now.')
        return (
            pd.read_csv(PROJECT_DIR / 'facial_age_manifest.csv'),
            pd.read_csv(PROJECT_DIR / 'train_manifest.csv'),
            pd.read_csv(PROJECT_DIR / 'val_manifest.csv'),
            pd.read_csv(PROJECT_DIR / 'test_manifest.csv'),
            pd.read_csv(PROJECT_DIR / 'block1_autoencoder_manifest.csv'),
            pd.read_csv(PROJECT_DIR / 'block2_classification_manifest.csv')
        )
    return build_manifests()

In [3]:
df, train_df, val_df, test_df, block1_df, block2_df = ensure_manifests()

Manifest files created successfully:
 - /content/facial_age_project/facial_age_manifest.csv
 - /content/facial_age_project/train_manifest.csv
 - /content/facial_age_project/val_manifest.csv
 - /content/facial_age_project/test_manifest.csv
 - /content/facial_age_project/block1_autoencoder_manifest.csv
 - /content/facial_age_project/block2_classification_manifest.csv


## 2. Training settings and class labels

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42



IMG_SIZE = 64
BATCH_SIZE = 16
EPOCHS = 2
MAX_BLOCK1_SAMPLES = 1200
MAX_BLOCK2_SAMPLES = 1500
MAX_VAL_SAMPLES = 500

age_group_order = ['child', 'teen', 'youth', 'adult', 'mature', 'senior']
group_to_id = {name: i for i, name in enumerate(age_group_order)}
NUM_CLASSES = len(age_group_order)


try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision enabled.')
except Exception:
    print('Mixed precision not enabled, continuing normally.')

print('GPU devices:', tf.config.list_physical_devices('GPU'))

block1_small = block1_df.sample(min(len(block1_df), MAX_BLOCK1_SAMPLES), random_state=SEED).reset_index(drop=True)
block2_small = block2_df.sample(min(len(block2_df), MAX_BLOCK2_SAMPLES), random_state=SEED).reset_index(drop=True)
val_small = test_df.sample(min(len(test_df), MAX_VAL_SAMPLES), random_state=SEED).reset_index(drop=True)

print('Autoencoder training images:', len(block1_small))
print('Classification training images:', len(block2_small))
print('Validation images:', len(val_small))


Mixed precision enabled.
GPU devices: []
Autoencoder training images: 1200
Classification training images: 1500
Validation images: 500


## 3. Data pipelines for Block 1 and Block 2

In [5]:
AUTOTUNE = tf.data.AUTOTUNE

# Images are resized to a compact shape to keep Part 3 practical.
def decode_rgb(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.float32) / 255.0

# For the autoencoder, the input and target are the same image.
# The image is decoded once and then reused as both x and y, which avoids unnecessary file reading.
def make_autoencoder_pair(path):
    img = decode_rgb(path)
    return img, img

def make_autoencoder_ds(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(frame['image_path'].values)
    if shuffle:
        ds = ds.shuffle(min(len(frame), 1000), seed=SEED)
    ds = ds.map(make_autoencoder_pair, num_parallel_calls=AUTOTUNE)
    return ds.cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)

def make_cls_ds(frame, shuffle=False):
    labels = frame['age_group'].map(group_to_id).astype('int32').values
    ds = tf.data.Dataset.from_tensor_slices((frame['image_path'].values, labels))
    if shuffle:
        ds = ds.shuffle(min(len(frame), 1000), seed=SEED)
    ds = ds.map(lambda p, y: (decode_rgb(p), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)

ae_train = make_autoencoder_ds(block1_small, shuffle=True)
cls_train = make_cls_ds(block2_small, shuffle=True)
cls_val = make_cls_ds(val_small, shuffle=False)

print('Datasets are ready.')


Datasets are ready.


## 4. Autoencoder model

The autoencoder learns to compress and reconstruct face images. I use mean squared error because the output is an image reconstruction.

In [6]:
def build_autoencoder():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    encoded = layers.MaxPooling2D(name='encoder_output')(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(encoded)
    x = layers.UpSampling2D()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.UpSampling2D()(x)
    outputs = layers.Conv2D(3, 3, padding='same', activation='sigmoid', dtype='float32')(x)

    model = keras.Model(inputs, outputs, name='simple_autoencoder')
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

autoencoder = build_autoencoder()
autoencoder.summary()

Model: "simple_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_output (MaxPooling2D)   │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 64, 3)      │           867 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,651 (295.51 KB)

 Trainable params: 75,651 (295.51 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Train and save the autoencoder

The autoencoder still learns to reconstruct Block 1 images, and the training history is available for plotting and discussion in the report. The smaller 64×64 input is a practical choice because reconstruction models become slow when every pixel has to be predicted.

In [7]:
# The autoencoder is trained for a short exploratory run.
# This is enough to demonstrate reconstruction learning and produce curves for discussion.
history_ae = autoencoder.fit(ae_train, epochs=EPOCHS, verbose=2)

MODEL_DIR = PROJECT_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)
autoencoder.save(MODEL_DIR / 'part3_autoencoder.keras')
print('Autoencoder saved.')


Epoch 1/2
75/75 - 684s - 9s/step - loss: 0.0220 - mae: 0.1084
Epoch 2/2
75/75 - 666s - 9s/step - loss: 0.0061 - mae: 0.0560
Autoencoder saved.


## 6. Transfer learning from the autoencoder encoder

Here I reuse the encoder part of the autoencoder and attach a classifier for age groups.

In [8]:
def build_autoencoder_transfer_classifier(autoencoder):
    encoder_output = autoencoder.get_layer('encoder_output').output
    encoder = keras.Model(autoencoder.input, encoder_output, name='reused_encoder')
    encoder.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = encoder(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = keras.Model(inputs, outputs, name='autoencoder_transfer_classifier')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

transfer_model = build_autoencoder_transfer_classifier(autoencoder)
transfer_model.summary()

Model: "autoencoder_transfer_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reused_encoder (Functional)     │ (None, 16, 16, 64)     │        19,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,486 (111.27 KB)

 Trainable params: 9,094 (35.52 KB)

 Non-trainable params: 19,392 (75.75 KB)

## 5. Train and save the autoencoder

The autoencoder still learns to reconstruct Block 1 images, and the training history is available for plotting and discussion in the report. The smaller 64×64 input is a practical choice because reconstruction models become slow when every pixel has to be predicted.

In [9]:
# This classifier reuses the encoder learned by the autoencoder.
# The goal is to check whether the reconstruction features help with age-group classification.
history_transfer = transfer_model.fit(cls_train, validation_data=cls_val, epochs=EPOCHS, verbose=2)
transfer_model.save(MODEL_DIR / 'part3_autoencoder_transfer_classifier.keras')
print('Transfer classifier saved.')


Epoch 1/2
94/94 - 13s - 135ms/step - accuracy: 0.3267 - loss: 1.6895 - val_accuracy: 0.3240 - val_loss: 1.6921
Epoch 2/2
94/94 - 16s - 174ms/step - accuracy: 0.3447 - loss: 1.6663 - val_accuracy: 0.3240 - val_loss: 1.6821
Transfer classifier saved.


## 8. Generic ImageNet backbone model

This model uses a general image-processing backbone pretrained on ImageNet, not an age-specific model. That matches the assignment requirement.

In [10]:
base = keras.applications.MobileNetV2(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

backbone_model = keras.Model(inputs, outputs, name='mobilenetv2_backbone_classifier')
backbone_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
backbone_model.summary()

/tmp/ipykernel_28090/3236883253.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "mobilenetv2_backbone_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 9. Train and save the backbone model

In [11]:
# MobileNetV2 is used as the generic ImageNet backbone comparison.
# It is intentionally not an age-specific model, which keeps this aligned with the requirement.
history_backbone = backbone_model.fit(cls_train, validation_data=cls_val, epochs=EPOCHS, verbose=2)
backbone_model.save(MODEL_DIR / 'part3_backbone_classifier.keras')
print('Backbone classifier saved.')


Epoch 1/2
94/94 - 45s - 477ms/step - accuracy: 0.4833 - loss: 1.4509 - val_accuracy: 0.5740 - val_loss: 1.1261
Epoch 2/2
94/94 - 36s - 383ms/step - accuracy: 0.5967 - loss: 1.0081 - val_accuracy: 0.5680 - val_loss: 1.0574
Backbone classifier saved.


## 10. Save Part 3 comparison table

In [12]:
comparison = pd.DataFrame([
    {'model': 'autoencoder_transfer_classifier', 'final_val_accuracy': history_transfer.history['val_accuracy'][-1]},
    {'model': 'mobilenetv2_backbone_classifier', 'final_val_accuracy': history_backbone.history['val_accuracy'][-1]},
])
comparison.to_csv(PROJECT_DIR / 'part3_model_comparison.csv', index=False)
comparison

,model,final_val_accuracy
0,autoencoder_transfer_classifier,0.324
1,mobilenetv2_backbone_classifier,0.568
